In [4]:
%matplotlib qt5

import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL
from matplotlib.lines import Line2D
from dataclasses import dataclass

In [5]:
## Paths used for loading and saving data
path = Path("D:\\Datasets\\SPED_tilt_series\\SPED_tilt_series")
path_calibrated = Path("D:\\Datasets\\SPED_tilt_series\\series_calibrated")
path_target = Path("D:\\Datasets\\SPED_tilt_series\\target_areas")
path_reference = Path("D:\\Datasets\\SPED_tilt_series\\reference_areas")

In [ ]:
# The scaling factor used was determined by manually calibrating one of the datasets and extracting the scale factor from the calibrated data. #
def preprocess_data(path, path_calibrated, path_target, path_reference, scale_factor):
    files = sorted(path.glob("*.hspy"))
    for i in files:
        filename = i.stem
        number = filename.rsplit("_", 1)[-1]

        ## Calibrate the data ##
        load_i = hs.load(i)
        load_i.axes_manager[2].scale = scale_factor
        load_i.axes_manager[3].scale = scale_factor

        shifts = load_i.get_direct_beam_position(method="blur", sigma=5, half_square_width=10)
        linear_shifts = shifts.get_linear_plane()
        load_i_centered = load_i.center_direct_beam(shifts=linear_shifts, inplace=False)


        ## Select region of interest ##
        reference_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)
        target_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)

        load_i_centered.plot()

        reference_roi = reference_rectangle.interactive(load_i_centered)
        target_roi = target_rectangle.interactive(load_i_centered, color='C1')
        
        plt.show(block=True)

        reference_roi.save(path_reference / f"SPED_tilt_series_reference_{number}.hspy")
        target_roi.save(path_target / f"SPED_tilt_series_target_{number}.hspy")
        load_i_centered.save(path_calibrated / f"SPED_tilt_series_calibrated_{number}.hspy")

In [6]:
# Define the crystal structure for aluminum (Al)
a = 4.05
atoms = [Atom('Al', [0, 0, 0]), Atom('Al', [0.5, 0.5, 0]), Atom('Al', [0.5, 0, 0.5]), Atom('Al', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Al', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [12]:
@dataclass
class SPEDResult:
    results: object
    correlations: object
    best_single_phase: object   
    
    def ipf_correlation(self):
        template_indices_i = (self.results.data[:, 0]).astype('int16')
        orientations_i = orientations[template_indices_i]
        
        fig = plt.figure()
        ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)
        ax.scatter(orientations_i, c=self.correlations, cmap='inferno')
        ax.scatter(orientations_i[0], c="r", marker="x")
        
        fig_dp, ax_dp = plt.subplots()
        first = template_indices_i[0]
        
        sim_i = simulations.irot[first]
        pat = sim_i.get_diffraction_pattern(
            shape=(256, 256),        # pick your detector size
            sigma=2,                 # spot size in pixels
            calibration=0.01,        # real->pixel scale (tune to match your data)
            direct_beam_position=(128, 128),
            normalize=True,
        )
        ax_dp.imshow(pat, origin="lower")

    def ipf_matches(self):
        fig = self.single_phase.scatter(projection="ipf", return_figure=True, label=labels[0])
        plt.show()

In [47]:
target_files = sorted(path_target.glob("*.hspy"))
reference_files = sorted(path_reference.glob("*.hspy"))

def template(ref_data, target_data):
    ref_data.change_dtype("float32")
    target_data.change_dtype("float32")

    ny, nx = target_data.axes_manager.signal_shape
    center = (ny/2 - 0.5, nx/2 - 0.5)
    
    ref_data.calibration(center=center)
    target_data.calibration(center=center)
    
    target_azi = target_data.get_azimuthal_integral2d(npt=256)
    ref_azi = ref_data.get_azimuthal_integral2d(npt=256)
    
    target_results = target_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    ref_results = ref_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    
    target_single_phase = target_results.to_single_phase_orientations()
    reference_single_phase = ref_results.to_single_phase_orientations()
    
    target_correlation = target_results.data[:, 1]
    reference_correlation = ref_results.data[:, 1]
    
    target = SPEDResult(
        results=target_results, 
        correlations=target_correlation, 
        best_single_phase=target_single_phase)
    
    reference = SPEDResult(
        results=ref_results, 
        correlations=reference_correlation, 
        best_single_phase=reference_single_phase)
    
    return target, reference

In [48]:
misorientation_array = []

for target_file, reference_file in zip(target_files, reference_files):
    target_data = hs.load(target_file)
    reference_data = hs.load(reference_file)

    target_mean = target_data.mean((0, 1))
    reference_mean = reference_data.mean((0, 1))

    target_result, reference_result = template(reference_mean, target_mean)

    misorientation_array.append([target_result, reference_result])


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5487)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5492)


  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5487)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5492)


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

In [63]:
difference_array = np.zeros(np.shape(misorientation_array)[0])
for i, misorientation in enumerate(misorientation_array):
    difference = misorientation[0].best_single_phase[0] - misorientation[1].best_single_phase[0]
    difference_array[i] = difference.angle
    if i > 42:
        if abs(difference_array[i] - difference_array[i-1]) > 0.10:

            for k in range(0, 100):
                for j in range(1, 100):
                    alternative = misorientation[0].best_single_phase[k] - misorientation[1].best_single_phase[j]
                    
                    if abs(alternative.angle - difference_array[i-1]) < 0.10:
                        difference_array[i] = alternative.angle
                        break
                if abs(alternative.angle - difference_array[i-1]) < 0.10:
                    break


number = np.arange(1, len(angles_array) + 1)

plt.figure()
plt.plot(number, difference_array, marker='o')
plt.show()

C:\Users\denis\AppData\Local\Temp\ipykernel_26688\1672533119.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  difference_array[i] = difference.angle
C:\Users\denis\AppData\Local\Temp\ipykernel_26688\1672533119.py:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  difference_array[i] = alternative.angle


In [ ]:
difference_array2 = np.zeros(np.shape(misorientation_array)[0])
for l, misorientation in enumerate(misorientation_array):
    difference = misorientation[0].best_single_phase[0] - misorientation[1].best_single_phase[0]
    difference_array2[l] = difference.angle

for m in range(len(difference_array2)):
    result_fix = misorientation_array[m]
    if m > 1:
        if abs(difference_array2[m+1] - difference_array2[m-1]) < 0.07:
            if difference_array2[m] - difference_array2[m-1] > 0.10:
                for n in range(0, 100):
                    for o in range(1, 100):
                        alternative = result_fix[0].best_single_phase[k] - result_fix[1].best_single_phase[j]
                    
                        if abs(alternative.angle - difference_array2[m-1]) < 0.10:
                            difference_array2[m] = alternative.angle
                            break
                    if abs(alternative.angle - difference_array2[m-1]) < 0.10:
                        break


number = np.arange(1, len(angles_array) + 1)

plt.figure()
plt.plot(number, difference_array2, marker='o')
plt.show()

C:\Users\denis\AppData\Local\Temp\ipykernel_26688\837866934.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  difference_array2[l] = difference.angle


KeyboardInterrupt: 